# ROOKHIDE — Full Performance Benchmark 🏰

Test the entire encode → decode pipeline across **light** (32 B – 10 KB) and **heavy** (50 KB – 2 MB) files.  
Measure execution time, throughput (bytes/sec, moves/sec), memory usage, and verify byte-perfect round-trips.  
Visualize everything with publication-quality graphs.

## 1. Import Required Libraries

In [ ]:
import os, sys, time, shutil, tracemalloc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# Ensure project root is on path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd()))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from encode import encode
from decode import decode

# Plotting style
sns.set_theme(style="darkgrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.figsize": (12, 5),
    "figure.dpi": 120,
    "savefig.dpi": 150,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

# Directories
TEST_DIR   = os.path.join(PROJECT_ROOT, "_bench_files")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "_bench_outputs")
os.makedirs(TEST_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Test files   : {TEST_DIR}")
print(f"Output files : {OUTPUT_DIR}")
print("Ready ✓")

## 2. Define File Generation Utilities & Test Matrix

In [ ]:
def generate_test_file(size_bytes: int, path: str):
    """Generate a file of exactly `size_bytes` random bytes."""
    data = os.urandom(size_bytes)
    with open(path, "wb") as f:
        f.write(data)
    return data

def human_size(nbytes: int) -> str:
    for unit in ("B", "KB", "MB"):
        if nbytes < 1024:
            return f"{nbytes:.0f} {unit}" if unit == "B" else f"{nbytes:.1f} {unit}"
        nbytes /= 1024
    return f"{nbytes:.1f} GB"

# ── Test matrix ──────────────────────────────────────────────────────────────
LIGHT_SIZES = [32, 64, 128, 256, 512, 1024, 2048, 5120, 10240]          # 32 B → 10 KB
HEAVY_SIZES = [51200, 102400, 262144, 524288, 1048576, 2097152]          # 50 KB → 2 MB

ALL_SIZES = LIGHT_SIZES + HEAVY_SIZES
LABELS    = [human_size(s) for s in ALL_SIZES]

print(f"Light files ({len(LIGHT_SIZES)}): {', '.join(human_size(s) for s in LIGHT_SIZES)}")
print(f"Heavy files ({len(HEAVY_SIZES)}): {', '.join(human_size(s) for s in HEAVY_SIZES)}")
print(f"Total tests : {len(ALL_SIZES)}")

## 3. Generate Light Test Files

In [ ]:
light_files = {}
print("Generating light test files:")
for size in LIGHT_SIZES:
    fname = f"test_{size}B.bin"
    fpath = os.path.join(TEST_DIR, fname)
    generate_test_file(size, fpath)
    light_files[size] = fpath
    print(f"  ✓ {fname:20s}  {human_size(size):>8s}")
print(f"\n{len(light_files)} light files ready.")

## 4. Generate Heavy Test Files

In [ ]:
heavy_files = {}
print("Generating heavy test files:")
for size in HEAVY_SIZES:
    fname = f"test_{size}B.bin"
    fpath = os.path.join(TEST_DIR, fname)
    generate_test_file(size, fpath)
    heavy_files[size] = fpath
    print(f"  ✓ {fname:20s}  {human_size(size):>8s}")
print(f"\n{len(heavy_files)} heavy files ready.")

## 5. Run Encode / Decode on Light Files

In [ ]:
def benchmark_file(size: int, input_path: str, label: str) -> dict:
    """Encode then decode a single file, returning all metrics."""
    pgn_path     = os.path.join(OUTPUT_DIR, f"enc_{size}.pgn")
    decoded_path = os.path.join(OUTPUT_DIR, f"dec_{size}.bin")

    original = open(input_path, "rb").read()

    # ── Encode ───────────────────────────────────────────────────
    tracemalloc.start()
    t0 = time.perf_counter()
    encode(input_path, pgn_path)
    enc_time = time.perf_counter() - t0
    enc_mem_peak = tracemalloc.get_traced_memory()[1]   # peak bytes
    tracemalloc.stop()

    pgn_size = os.path.getsize(pgn_path)

    # ── Decode ───────────────────────────────────────────────────
    tracemalloc.start()
    t0 = time.perf_counter()
    decode(pgn_path, decoded_path)
    dec_time = time.perf_counter() - t0
    dec_mem_peak = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()

    decoded = open(decoded_path, "rb").read()
    match = (original == decoded)

    return {
        "label":           label,
        "category":        "Light" if size <= 10240 else "Heavy",
        "size_bytes":      size,
        "enc_time_s":      round(enc_time, 4),
        "dec_time_s":      round(dec_time, 4),
        "total_time_s":    round(enc_time + dec_time, 4),
        "pgn_size_bytes":  pgn_size,
        "expansion_ratio": round(pgn_size / size, 2),
        "enc_throughput_KBs": round(size / 1024 / max(enc_time, 1e-6), 2),
        "dec_throughput_KBs": round(size / 1024 / max(dec_time, 1e-6), 2),
        "enc_mem_peak_MB": round(enc_mem_peak / 1024 / 1024, 2),
        "dec_mem_peak_MB": round(dec_mem_peak / 1024 / 1024, 2),
        "match":           match,
    }

# ── Run light files ──────────────────────────────────────────────────────
results = []
print("Running LIGHT file benchmarks…")
for size, fpath in light_files.items():
    lbl = human_size(size)
    print(f"  ⏳ {lbl:>8s} … ", end="", flush=True)
    r = benchmark_file(size, fpath, lbl)
    results.append(r)
    status = "✓ MATCH" if r["match"] else "✗ MISMATCH"
    print(f"enc {r['enc_time_s']:.3f}s  dec {r['dec_time_s']:.3f}s  {status}")
print("Light benchmarks complete.")

## 6. Run Encode / Decode on Heavy Files

In [ ]:
print("Running HEAVY file benchmarks (this may take a few minutes)…")
for size, fpath in heavy_files.items():
    lbl = human_size(size)
    print(f"  ⏳ {lbl:>8s} … ", end="", flush=True)
    r = benchmark_file(size, fpath, lbl)
    results.append(r)
    status = "✓ MATCH" if r["match"] else "✗ MISMATCH"
    print(f"enc {r['enc_time_s']:.3f}s  dec {r['dec_time_s']:.3f}s  {status}")
print("Heavy benchmarks complete.")

## 7. Aggregate Results — Summary Table

In [ ]:
df = pd.DataFrame(results)

# Add human-readable size column (ordered)
df["size_label"] = pd.Categorical(df["label"], categories=[human_size(s) for s in ALL_SIZES], ordered=True)

# Display summary
display_cols = ["label", "category", "size_bytes", "enc_time_s", "dec_time_s",
                "total_time_s", "pgn_size_bytes", "expansion_ratio",
                "enc_throughput_KBs", "dec_throughput_KBs",
                "enc_mem_peak_MB", "dec_mem_peak_MB", "match"]
print("="*120)
print("ROOKHIDE BENCHMARK RESULTS")
print("="*120)
df[display_cols].to_string(index=False)
df[display_cols]

## 8. Execution Time Comparison — Light vs Heavy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Left: Grouped bar chart (encode vs decode) ──────────────────────────
x = np.arange(len(df))
w = 0.35
axes[0].bar(x - w/2, df["enc_time_s"], w, label="Encode", color="#e94560", edgecolor="white")
axes[0].bar(x + w/2, df["dec_time_s"], w, label="Decode", color="#0f3460", edgecolor="white")
axes[0].set_xticks(x)
axes[0].set_xticklabels(df["label"], rotation=45, ha="right")
axes[0].set_ylabel("Time (seconds)")
axes[0].set_title("Encode vs Decode Time by File Size")
axes[0].legend()
axes[0].set_yscale("log")
axes[0].yaxis.set_major_formatter(ticker.ScalarFormatter())

# ── Right: Stacked total time, colored by category ──────────────────────
colors = ["#16c79a" if c == "Light" else "#ff6b6b" for c in df["category"]]
axes[1].bar(x, df["total_time_s"], color=colors, edgecolor="white")
axes[1].set_xticks(x)
axes[1].set_xticklabels(df["label"], rotation=45, ha="right")
axes[1].set_ylabel("Total Round-Trip Time (seconds)")
axes[1].set_title("Total Time: Light (green) vs Heavy (red)")
axes[1].set_yscale("log")
axes[1].yaxis.set_major_formatter(ticker.ScalarFormatter())

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "01_time_comparison.png"), bbox_inches="tight")
plt.show()

## 9. Memory Usage Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

x = np.arange(len(df))
w = 0.35

# ── Left: Peak memory bar chart ─────────────────────────────────────────
axes[0].bar(x - w/2, df["enc_mem_peak_MB"], w, label="Encode Peak", color="#e94560")
axes[0].bar(x + w/2, df["dec_mem_peak_MB"], w, label="Decode Peak", color="#0f3460")
axes[0].set_xticks(x)
axes[0].set_xticklabels(df["label"], rotation=45, ha="right")
axes[0].set_ylabel("Peak Memory (MB)")
axes[0].set_title("Peak Memory Usage: Encode vs Decode")
axes[0].legend()

# ── Right: Memory vs file size (line plot) ───────────────────────────────
axes[1].plot(df["size_bytes"] / 1024, df["enc_mem_peak_MB"], "o-", color="#e94560", label="Encode", linewidth=2, markersize=6)
axes[1].plot(df["size_bytes"] / 1024, df["dec_mem_peak_MB"], "s-", color="#0f3460", label="Decode", linewidth=2, markersize=6)
axes[1].set_xlabel("Input File Size (KB)")
axes[1].set_ylabel("Peak Memory (MB)")
axes[1].set_title("Memory Scaling with File Size")
axes[1].set_xscale("log")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "02_memory_comparison.png"), bbox_inches="tight")
plt.show()

## 10. Throughput & File Size Scaling

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sizes_kb = df["size_bytes"] / 1024

# ── Left: Throughput (KB/s) vs file size ─────────────────────────────────
axes[0].plot(sizes_kb, df["enc_throughput_KBs"], "o-", color="#e94560", label="Encode", linewidth=2.5, markersize=7)
axes[0].plot(sizes_kb, df["dec_throughput_KBs"], "s-", color="#0f3460", label="Decode", linewidth=2.5, markersize=7)
axes[0].set_xlabel("Input File Size (KB)")
axes[0].set_ylabel("Throughput (KB/s)")
axes[0].set_title("Throughput vs File Size")
axes[0].set_xscale("log")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ── Right: Time scaling (log-log) ───────────────────────────────────────
axes[1].loglog(sizes_kb, df["enc_time_s"], "o-", color="#e94560", label="Encode", linewidth=2.5, markersize=7)
axes[1].loglog(sizes_kb, df["dec_time_s"], "s-", color="#0f3460", label="Decode", linewidth=2.5, markersize=7)
axes[1].loglog(sizes_kb, df["total_time_s"], "D--", color="#f5a623", label="Total", linewidth=2, markersize=6)
# Reference line: perfect linear scaling
ref_x = np.array([sizes_kb.min(), sizes_kb.max()])
ref_y = df["total_time_s"].iloc[0] * (ref_x / ref_x[0])
axes[1].plot(ref_x, ref_y, ":", color="gray", alpha=0.6, label="Linear ∝ size")
axes[1].set_xlabel("Input File Size (KB)")
axes[1].set_ylabel("Time (seconds)")
axes[1].set_title("Scaling Behaviour (log–log)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "03_throughput_scaling.png"), bbox_inches="tight")
plt.show()

## 11. Per-File Processing Distribution (Box + Violin)

In [ ]:
# Melt encode/decode times for violin & box plots
melted = df.melt(id_vars=["label", "category", "size_label"],
                 value_vars=["enc_time_s", "dec_time_s"],
                 var_name="phase", value_name="time_s")
melted["phase"] = melted["phase"].map({"enc_time_s": "Encode", "dec_time_s": "Decode"})

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Left: Box plot by category ───────────────────────────────────────────
sns.boxplot(data=melted, x="category", y="time_s", hue="phase",
            palette={"Encode": "#e94560", "Decode": "#0f3460"}, ax=axes[0])
axes[0].set_title("Processing Time Distribution: Light vs Heavy")
axes[0].set_ylabel("Time (seconds)")
axes[0].set_xlabel("")

# ── Right: Violin plot ──────────────────────────────────────────────────
sns.violinplot(data=melted, x="category", y="time_s", hue="phase", split=True,
               palette={"Encode": "#e94560", "Decode": "#0f3460"}, ax=axes[1], inner="quart")
axes[1].set_title("Time Distribution (Violin)")
axes[1].set_ylabel("Time (seconds)")
axes[1].set_xlabel("")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "04_distribution.png"), bbox_inches="tight")
plt.show()

## 12. PGN Expansion Ratio & Encode/Decode Speed Ratio

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

x = np.arange(len(df))

# ── Left: PGN Expansion Ratio ───────────────────────────────────────────
colors = ["#16c79a" if c == "Light" else "#ff6b6b" for c in df["category"]]
axes[0].bar(x, df["expansion_ratio"], color=colors, edgecolor="white")
axes[0].axhline(y=df["expansion_ratio"].mean(), color="gray", linestyle="--", alpha=0.7,
                label=f'Mean: {df["expansion_ratio"].mean():.1f}×')
axes[0].set_xticks(x)
axes[0].set_xticklabels(df["label"], rotation=45, ha="right")
axes[0].set_ylabel("PGN Size / Original Size")
axes[0].set_title("PGN Expansion Ratio (smaller = more efficient)")
axes[0].legend()

# ── Right: Decode/Encode speed ratio ────────────────────────────────────
speed_ratio = df["dec_throughput_KBs"] / df["enc_throughput_KBs"].replace(0, np.nan)
axes[1].bar(x, speed_ratio, color="#f5a623", edgecolor="white")
axes[1].axhline(y=1.0, color="gray", linestyle="--", alpha=0.7, label="1:1 ratio")
axes[1].set_xticks(x)
axes[1].set_xticklabels(df["label"], rotation=45, ha="right")
axes[1].set_ylabel("Decode / Encode Speed Ratio")
axes[1].set_title("Decode is Faster Than Encode When Ratio > 1")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "05_expansion_speed_ratio.png"), bbox_inches="tight")
plt.show()

## 13. Summary Heatmap & Final Statistics

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────
numeric_cols = ["size_bytes", "enc_time_s", "dec_time_s", "total_time_s",
                "pgn_size_bytes", "expansion_ratio",
                "enc_throughput_KBs", "dec_throughput_KBs",
                "enc_mem_peak_MB", "dec_mem_peak_MB"]
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdYlGn",
            center=0, square=True, linewidths=0.5, ax=ax,
            vmin=-1, vmax=1,
            xticklabels=[c.replace("_", " ").title() for c in numeric_cols],
            yticklabels=[c.replace("_", " ").title() for c in numeric_cols])
ax.set_title("Metric Correlation Heatmap", fontsize=15, pad=15)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "06_correlation_heatmap.png"), bbox_inches="tight")
plt.show()

In [ ]:
# ── Summary statistics by category ────────────────────────────────────────
summary = df.groupby("category").agg(
    files=("label", "count"),
    avg_enc_s=("enc_time_s", "mean"),
    avg_dec_s=("dec_time_s", "mean"),
    avg_total_s=("total_time_s", "mean"),
    max_total_s=("total_time_s", "max"),
    avg_enc_KBs=("enc_throughput_KBs", "mean"),
    avg_dec_KBs=("dec_throughput_KBs", "mean"),
    avg_expansion=("expansion_ratio", "mean"),
    avg_enc_mem_MB=("enc_mem_peak_MB", "mean"),
    avg_dec_mem_MB=("dec_mem_peak_MB", "mean"),
    all_match=("match", "all"),
).round(3)

print("="*90)
print("SUMMARY BY CATEGORY")
print("="*90)
display(summary)

# ── Overall stats ────────────────────────────────────────────────────────
all_match = df["match"].all()
print(f"\n{'='*90}")
print(f"OVERALL: {len(df)} files tested | All round-trips match: {'✓ YES' if all_match else '✗ NO'}")
print(f"Largest file: {human_size(df['size_bytes'].max())} → "
      f"Encode {df.loc[df['size_bytes'].idxmax(), 'enc_time_s']:.2f}s, "
      f"Decode {df.loc[df['size_bytes'].idxmax(), 'dec_time_s']:.2f}s, "
      f"Total {df.loc[df['size_bytes'].idxmax(), 'total_time_s']:.2f}s")
print(f"Peak encode throughput: {df['enc_throughput_KBs'].max():.0f} KB/s")
print(f"Peak decode throughput: {df['dec_throughput_KBs'].max():.0f} KB/s")
print(f"{'='*90}")

In [ ]:
# ── Cleanup temp files ────────────────────────────────────────────────────
shutil.rmtree(TEST_DIR, ignore_errors=True)
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
print("Temporary benchmark files cleaned up ✓")